In [2]:
# import libraries
import numpy as np
import pandas as pd

In [5]:
# read in the csv file
df = pd.read_csv('../data/raw/sales_data.csv')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  50000 non-null  object 
 1   date            50000 non-null  object 
 2   product_id      50000 non-null  object 
 3   store_id        50000 non-null  object 
 4   customer_id     48156 non-null  object 
 5   quantity        50000 non-null  int64  
 6   discount        47417 non-null  float64
 7   returned        50000 non-null  int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 3.1+ MB


In [6]:
# check for null values
df.isnull().sum()

transaction_id       0
date                 0
product_id           0
store_id             0
customer_id       1844
quantity             0
discount          2583
returned             0
dtype: int64

In [8]:
# handle missing customer ids
# no customer id = guest
df['customer_id'] = df['customer_id'].fillna("Guest")
df.isnull().sum()

transaction_id       0
date                 0
product_id           0
store_id             0
customer_id          0
quantity             0
discount          2583
returned             0
dtype: int64

In [9]:
# handle missing discount values
# null discount = 0%/no discount
df['discount'] = df['discount'].fillna(0)
df.isnull().sum()

transaction_id    0
date              0
product_id        0
store_id          0
customer_id       0
quantity          0
discount          0
returned          0
dtype: int64

In [13]:
# transaction_id
# check that there are no duplicate transaction ids
df['transaction_id'].duplicated().sum()

0

In [ ]:
# date
# check data type
df['date'].dtype

# change to datetime type
df['date'] = pd.to_datetime(df['date'])
df['date'].head(10)

dtype('<M8[ns]')

In [ ]:
# product_id
# check product id data type
df['product_id'].dtype

# make sure all product ids exist in the cleaned product table
product_df = pd.read_csv('../data/cleaned/product_data_cleaned.csv')
# return product ids in sales and not in cleaned product table
missing_ids = df[~df['product_id'].isin(product_df['product_id'])]
# get the missing ids
missing_ids['product_id'].unique()

# only one missing id from the product table: P999999
# check what these rows look like
df[df['product_id'] == "P999999"].head(10)

# keep the 200 rows
# note this finding in README


,transaction_id,date,product_id,store_id,customer_id,quantity,discount,returned
79,T0000080,2023-09-16,P999999,S003,C020459,4,0.0,0
847,T0000848,2022-05-10,P999999,S003,C018177,3,0.0,0
1270,T0001271,2023-03-11,P999999,S005,C001458,4,0.0,0
1470,T0001471,2022-08-26,P999999,S003,C023479,1,0.0,0
1555,T0001556,2022-06-01,P999999,S005,C018164,3,0.0,0
2029,T0002030,2023-03-05,P999999,S004,C005598,1,0.2,0
2066,T0002067,2022-05-18,P999999,S003,C015332,1,0.2,0
2300,T0002301,2023-10-21,P999999,S005,Guest,2,0.0,0
2327,T0002328,2022-02-10,P999999,S005,C006760,2,0.0,1
2336,T0002337,2024-02-22,P999999,S003,C024766,1,0.0,0


In [ ]:
# store_id
# check store id data type
df['store_id'].dtype

# check there are only 5 stores
df['store_id'].value_counts()

# 200 rows with unexpected store id S999
# check overlap with 200 missing product id P999999
missing_product = df[df['product_id'] == "P999999"]
missing_store = df[df['store_id'] == "S999"]

# get number of transactions with missing store and product
len(missing_product), len(missing_store)
len(set(missing_product['transaction_id']) & set(missing_store['transaction_id']))

# only 1 overlapping transaction
# keep missing store rows, but note in README

1

In [ ]:
# customer_id
# check customer id data type
df['customer_id'].dtype

# make sure all customer ids exist in the cleaned customer table
customer_df = pd.read_csv('../data/cleaned/customer_data_cleaned.csv')
# return customer ids in sales and not in cleaned customer table
missing_c_ids = df[~df['customer_id'].isin(customer_df['customer_id'])]
# get the missing ids (see if there are any that aren't "guests")
missing_c_ids['customer_id'].unique()

# "Guest" is the only id missing from the table

array(['Guest'], dtype=object)

In [45]:
# quantity
# check quantity data type
df['quantity'].dtype

# make sure quantity range is realistic
df['quantity'].describe()

count    50000.000000
mean         2.506480
std          1.119028
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          4.000000
Name: quantity, dtype: float64

In [47]:
# discount
# check data type
df['discount'].dtype

# make sure discount range is realistic
df['discount'].describe()

count    50000.000000
mean         0.055124
std          0.086655
min          0.000000
25%          0.000000
50%          0.000000
75%          0.100000
max          0.300000
Name: discount, dtype: float64

In [ ]:
# returned
# check data type
df['returned'].dtype

# change data type from int to boolean
df['returned'] = df['returned'].astype('bool')
df['returned'].dtype

# check there are only true or false values
df['returned'].value_counts()


returned
False    45041
True      4959
Name: count, dtype: int64

In [52]:
# write the cleaned data to a new csv file
df.to_csv('../data/cleaned/sales_data_cleaned.csv')